# Intermitencia de demanda semanal por sucursal-producto
Este cuaderno usa el mismo constructor `W-SUN` del experimento ML sobre el CSV maestro. Se incluyen todas las combinaciones observadas, sin eliminar series por intermitencia ni por falta de lags.

Se rellenan únicamente semanas interiores sin transacciones, bajo el supuesto de continuidad de captura. Un NaN de origen provoca un error y no se convierte en cero. Las combinaciones nunca observadas no se inventan. Los extremos de cada serie pueden ser semanas parciales; cajas cero puede coexistir con ventas en unidades.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
sys.path.insert(0, str(Path('..').resolve()))
from src.preprocessing.demanda_semanal import construir_semanal, resumir_intermitencia

df = pd.read_csv('../datasets/dataset_maestro_dashboard.csv')
semanal = construir_semanal(df)
df_intermitencia = resumir_intermitencia(semanal)
salida = Path('../resultados/semanal')
salida.mkdir(parents=True, exist_ok=True)
print('Combinaciones potenciales:', df.sucursal.nunique() * df.producto.nunique())
print('Combinaciones observadas:', len(df_intermitencia))
print('Observaciones semanales:', len(semanal))
print('Porcentaje global de observaciones cero:', semanal.cantidad.eq(0).mean() * 100)
print('El porcentaje global usa todas las semanas, no un promedio de porcentajes por serie.')
with pd.option_context('display.max_rows', None):
    display(df_intermitencia.sort_values('porcentaje_demanda_cero', ascending=False))
df_intermitencia.to_csv(salida / 'intermitencia_semanal.csv', index=False)

,sucursal,producto,semanas_totales,semanas_demanda_cero,porcentaje_demanda_cero,demanda_promedio_semanal
5,1,IBUPROFENO TABX800,133,133,100.000000,0.000000
34,3,LOPERAMIDA 2MG,131,131,100.000000,0.000000
39,3,OMEPRAZOL 20MG,27,27,100.000000,0.000000
43,4,AZITROMICINA 500MG,135,135,100.000000,0.000000
47,4,IBUPROFENO TABX800,133,133,100.000000,0.000000
...,...,...,...,...,...,...
225,17,VITAMINA C+ZINC+D3,23,8,34.782609,1.217391
27,2,VITAMINA C+ZINC+D3,133,42,31.578947,1.526316
203,15,VITAMINA C+ZINC+D3,86,23,26.744186,1.441860
68,5,VITAMINA C+ZINC+D3,134,33,24.626866,1.768657


In [ ]:
productos_interes = ['ENALAPRIL 10MG', 'METFORMINA 850MG', 'LOSARTAN 50MG']
for producto in productos_interes:
    caso = df_intermitencia[df_intermitencia.producto.str.upper().eq(producto)].sort_values('sucursal')
    print(producto, '| sucursales observadas:', len(caso))
    display(caso)
    if not caso.empty:
        caso.plot.bar(x='sucursal', y='porcentaje_demanda_cero', legend=False, figsize=(10, 3))
        plt.title(f'{producto}: semanas con cajas cero')
        plt.ylabel('Porcentaje de semanas')
        plt.ylim(0, 100)
        plt.tight_layout()
        plt.show()

,sucursal,producto,semanas_totales,semanas_demanda_cero,porcentaje_demanda_cero,demanda_promedio_semanal
220,17,ENALAPRIL 10MG,35,35,100.000000,0.000000
228,18,ENALAPRIL 10MG,21,20,95.238095,0.047619
100,8,ENALAPRIL 10MG,133,126,94.736842,0.052632
236,19,ENALAPRIL 10MG,14,13,92.857143,0.071429
181,14,ENALAPRIL 10MG,92,85,92.391304,0.076087
141,11,ENALAPRIL 10MG,127,116,91.338583,0.094488
168,13,ENALAPRIL 10MG,103,93,90.291262,0.097087
87,7,ENALAPRIL 10MG,134,120,89.552239,0.104478
73,6,ENALAPRIL 10MG,135,120,88.888889,0.266667
194,15,ENALAPRIL 10MG,77,68,88.311688,0.116883
